# Practical 5: Facial Emotion Recognition using a CNN (FER-2013)

**Aim:** To build a CNN that recognises 7 facial emotions (angry, disgust, fear, happy, neutral, sad, surprise) from 48 x 48 grayscale face images of the FER-2013 dataset, handling class imbalance.

> Reconstructed code (the source PDF has no code). Not executed by the author - run cells in order (Kernel -> Restart & Run All) and check dataset paths in the first cells. Expected values: Test accuracy ~54-55% (human ~65%, SOTA ~76%).; Severely imbalanced dataset (disgust 436 vs happy 7,215): use class_weight='balanced' or augmentation.

## Cell 1 - Import libraries

**What it does:**
- `os` is used to count images per class folder. The rest are the usual deep-learning libraries.

*Expected output (illustrative):* TensorFlow version: 2.x.x

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

## Cell 2 - Set the dataset path and load images

**What it does:**
- Change `DATA_ROOT` to wherever you extracted FER-2013 (on Kaggle e.g. `/kaggle/input/fer2013`).
- `color_mode='grayscale'` loads each face as 1 channel (48x48x1). 10% of the training folder is kept for validation; the official `test` folder is the final test set.
- `shuffle=False` for the test set keeps image order fixed so predictions can be matched with true labels.
- Class names are read alphabetically from the folder names.

*Expected output (illustrative):* Found ~28,709 files belonging to 7 classes (train) ... ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

In [ ]:
DATA_ROOT = "fer2013"
train_dir = os.path.join(DATA_ROOT, "train")
test_dir = os.path.join(DATA_ROOT, "test")

IMG_SIZE = (48, 48)
BATCH_SIZE = 64

train_ds = keras.utils.image_dataset_from_directory(
    train_dir, color_mode="grayscale", image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, validation_split=0.1, subset="training", seed=42)

val_ds = keras.utils.image_dataset_from_directory(
    train_dir, color_mode="grayscale", image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, validation_split=0.1, subset="validation", seed=42)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir, color_mode="grayscale", image_size=IMG_SIZE,
    batch_size=BATCH_SIZE, shuffle=False)

class_names = train_ds.class_names
num_classes = len(class_names)
print(class_names)

## Cell 3 - Inspect class imbalance

**What it does:**
- Counts how many training images each emotion has and plots a bar chart.
- You will see Disgust (very few) vs Happy (very many) - the imbalance problem.

*Expected output (illustrative):* A bar chart: 'happy' is the tallest bar, 'disgust' is almost invisible. (Your PDF: disgust 436 vs happy 7,215 in the full dataset.)

In [ ]:
counts = {c: len(os.listdir(os.path.join(train_dir, c))) for c in class_names}
print(counts)

plt.figure(figsize=(8, 4))
plt.bar(list(counts.keys()), list(counts.values()), color="steelblue")
plt.title("Training images per emotion")
plt.ylabel("Number of images")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Cell 4 - Compute class weights

**What it does:**
- Implements the 'balanced' formula: weight = total_samples / (n_classes x samples_in_class).
- Rare classes get large weights (Disgust ~ 9), common classes get small weights (Happy ~ 0.6). During training, mistakes on rare classes then cost more.
- This is the Keras equivalent of `class_weight='balanced'` mentioned in your PDF.

*Expected output (illustrative):* A dictionary like {0: ~1.03, 1: ~9.4, 2: ~1.0, 3: ~0.57, 4: ~0.83, 5: ~0.83, 6: ~1.28}  (approximate).

In [ ]:
total = sum(counts.values())
class_weight = {
    i: total / (num_classes * counts[name])
    for i, name in enumerate(class_names)
}
print(class_weight)

## Cell 5 - Normalisation and augmentation layers

**What it does:**
- Faces are flipped, rotated and zoomed a little during training - important for a small, noisy dataset.
- `prefetch` overlaps data loading with training for speed. Pixel scaling to [0,1] is done inside the model with a Rescaling layer (next cell).

*Expected output (illustrative):* No output.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## Cell 6 - Build the CNN

**What it does:**
- `conv_block` = Conv2D -> BatchNorm -> ReLU -> MaxPool -> Dropout. BatchNorm is placed **before** ReLU (as in your PDF) because it stabilises the values entering the activation.
- `padding='same'` keeps the size unchanged before pooling; each MaxPool halves it: 48 -> 24 -> 12 -> 6.
- Filters grow 32 -> 64 -> 128: early layers see edges, deeper layers see facial parts.
- `Rescaling(1/255)` normalises pixels inside the model. Flatten gives 6 x 6 x 128 = 4,608 values -> Dense(256) -> Dropout(0.5) -> Dense(7, softmax).

*Expected output (illustrative):* model.summary(): output shapes (48,48,32) -> (24,24,32) -> (24,24,64) -> (12,12,64) -> (12,12,128) -> (6,6,128) -> (4608) -> (256) -> (7). Total params about 1.275 million.

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)
    return x

inputs = keras.Input(shape=(48, 48, 1))
x = layers.Rescaling(1.0 / 255)(inputs)
x = data_augmentation(x)
x = conv_block(x, 32)
x = conv_block(x, 64)
x = conv_block(x, 128)
x = layers.Flatten()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

## Cell 7 - Compile and train

**What it does:**
- `class_weight` applies the weights from Cell 4 to the loss.
- `EarlyStopping` stops when validation loss has not improved for 8 epochs and restores the best weights (prevents overfitting).
- `ReduceLROnPlateau` halves the learning rate if validation loss stalls for 3 epochs.

*Expected output (illustrative):* Up to 30 epochs of logs; usually stops earlier. Validation accuracy typically ends in the mid-50s %.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight,
    callbacks=callbacks,
)

## Cell 8 - Accuracy and loss graphs

**What it does:**
- Look for the gap between train and validation curves - a growing gap means overfitting.

*Expected output (illustrative):* Two graphs. Accuracy plateaus around 0.5-0.6 (this is a hard dataset).

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## Cell 9 - Test accuracy, confusion matrix, precision/recall/F1

**What it does:**
- Evaluates on the official test folder (~7,178 images).
- The confusion matrix shows which emotions are mixed up (fear<->sad, angry<->disgust). The report shows per-class precision, recall, F1 and **macro-F1**, which is more honest than accuracy on imbalanced data.

*Expected output (illustrative):* Test accuracy ~0.54 - 0.55 (from your PDF). Happy and Surprise have the best F1; Fear and Sad the worst.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print("Test accuracy:", round(test_acc, 4))

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix - FER-2013")
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

## Cell 10 - Predict single images and save the model

**What it does:**
- Shows 8 faces with true vs predicted emotion (green = correct).

*Expected output (illustrative):* 8 small face images with titles. Because accuracy is ~55%, expect several red titles.

In [ ]:
images, labels = next(iter(test_ds))
probs = model.predict(images, verbose=0)
preds = np.argmax(probs, axis=1)

plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i].numpy().squeeze().astype("uint8"), cmap="gray")
    ok = preds[i] == labels[i].numpy()
    plt.title("True: " + class_names[int(labels[i])] + "\nPred: " + class_names[preds[i]],
              color="green" if ok else "red", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

model.save("fer2013_cnn.keras")

## Result / Conclusion

A BatchNorm + Dropout CNN with class weights was trained on FER-2013 and reached ~54-55% test accuracy, close to what is achievable with a simple CNN on this noisy, imbalanced dataset.